# TP — Détection de beaconing C2

## Objectif

On dispose d'un dataframe contenant les logs de connexions sortantes sur un réseau d'entreprise.

Certains hôtes sont compromis et contactent régulièrement un serveur « Commande et contrôle » (C2). Le but du TD est de :
- identifier les hôtes compromis ;
- extraire les informations de compromission (IOC).

**Consignes :** complétez chaque fonction signalée par un encadré « ✏️ Exercice »,
puis exécutez la cellule de vérification qui la suit.

## Préambule : chargement des données

Le fichier `logs_reseau.csv` contient les connexions sortantes observées pendant
12 h (08:00 → 20:00) sur un réseau d'entreprise.

| colonne | description |
| --- | --- |
| `timestamp` | instant de la connexion |
| `src_ip` | hôte source (interne) |
| `dst_ip` | IP de destination |
| `domain` | domaine contacté (nom DNS) |
| `dst_port` | port de destination |
| `bytes_out` | octets envoyés |

In [ ]:
import statistics
from datetime import datetime, timedelta

import pandas as pd

df_logs_reseau = pd.read_csv("logs_reseau.csv", parse_dates=["timestamp"])
print(df_logs_reseau.shape)
df_logs_reseau.head()

In [ ]:
df_logs_reseau.info()

## 1. Beaconing « classique »

Un implant « beacon » rappelle son serveur C2 à intervalle **régulier**. Objectif :
repérer le couple (hôte, domaine) au rythme le plus régulier.

Pour cela, on calcule le « coefficient de variation » (CV) des intervalles de temps entre requêtes successives :

   - `CV ≈ 0` → intervalle quasi constant (beacon *ou* tâche planifiée) ;
   - `CV` élevé → navigation humaine, irrégulière.

Étant donnée une liste **triée** de timestamps de requêtes, le coefficient de variation se calcule ainsi :
- on construit la liste des intervalles de temps, **en secondes**, entre chaque couple de requêtes successives ;
- sur cette liste, `CV = écart-type / moyenne`.

**Méthode, pas à pas :**

1. **Regrouper en listes** : fabriquer un DataFrame où chaque ligne est un couple
   `(src_ip, domain)` et où l'on range ses timestamps dans une **liste** (triée).
2. **Transformer chaque liste en écarts** : une petite fonction Python
   (`get_diffs`) qui renvoie les temps inter-arrivées, en secondes.
3. **Résumer en CV** : calculer le coefficient de variation pour chacune de ces listes.

### 1.1 Regrouper les timestamps par couple (src_ip, domain)

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1</legend>

Complétez `timestamps_par_src_dest` : elle renvoie un dataframe indexé par `(src_ip, domain)`, avec une colonne `timestamps` contenant la liste triée des timestamps de connexion de chaque couple.

</fieldset>

In [ ]:
def timestamps_par_src_dest(df: pd.DataFrame) -> pd.DataFrame:
    """
    À partir du dataframe `logs_reseau`, retourner un dataframe qui a en index
    `(src_ip, domain)`, et une colonne `timestamps`, qui contient une liste triée
    de tous les timestamps de connexion associés à `(src_ip, domain)`
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df.sort_values("timestamp")
        .groupby(["src_ip", "domain"])
        .agg(timestamps=("timestamp", list))
    )
    # END

In [ ]:
ts_per_src_dest = timestamps_par_src_dest(df_logs_reseau)
nb_couples = df_logs_reseau[["src_ip", "domain"]].drop_duplicates().shape[0]
assert ts_per_src_dest.shape[0] == nb_couples
assert "timestamps" in ts_per_src_dest.columns
assert ts_per_src_dest["timestamps"].map(lambda ts: ts == sorted(ts)).all()
print("ex 1 OK :", nb_couples, "couples (src_ip, domain) distincts")

### 1.2 Calculer les écarts entre requêtes

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 2</legend>

Complétez `get_diffs` : elle renvoie, dans l'ordre chronologique, les écarts en secondes entre instants successifs d'une liste de timestamps (liste vide s'il y a moins de deux instants).

</fieldset>

In [ ]:
def get_diffs(timestamps: list[datetime]) -> list[float]:
    """Temps (en secondes) entre instants successifs, dans l'ordre chronologique.

    La liste est triée au besoin : aucune hypothèse sur l'ordre en entrée.
    """
    # BEGIN
    # raise NotImplementedError
    tri = sorted(timestamps)
    res = []
    for i in range(len(tri) - 1):
        res.append((tri[i + 1] - tri[i]).total_seconds())
    return res
    # END

In [ ]:
t0 = datetime(2026, 10, 14, 8, 0, 0)
assert get_diffs([t0, t0 + timedelta(seconds=60), t0 + timedelta(seconds=120)]) == [60.0, 60.0]
assert get_diffs([t0 + timedelta(seconds=120), t0, t0 + timedelta(seconds=60)]) == [60.0, 60.0]
assert get_diffs([t0]) == []
assert get_diffs([]) == []
print("ex 2 OK")

### 1.3 Résumer en coefficient de variation

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3</legend>

Complétez `cv_destinations` : elle calcule, à partir du dataframe de `timestamps_par_src_dest`, le nombre `n` d'écarts et le coefficient de variation `cv` de chaque couple `(src_ip, domain)`, en ignorant les couples ayant moins de `n_min` écarts.

</fieldset>

In [ ]:
def cv_destinations(df: pd.DataFrame, n_min: int = 20) -> pd.DataFrame:
    """Coefficient de variation des temps inter-arrivées par couple (src_ip, domain).

    Colonnes retournées : `src_ip`, `domain`, `n`, `cv`. On ignore les couples trop
    rares (moins de `n_min` connexions) : un CV sur 2 points ne veut rien dire.
    """
    # BEGIN
    # raise NotImplementedError
    ## Indice : le module `statistics` fournit `statistics.mean(liste)` et
    ## `statistics.stdev(liste)` : moyenne et écart-type d'une liste de nombres.
    df = df.copy()
    df["diffs"] = df["timestamps"].map(get_diffs)
    df["n"] = df["diffs"].map(len)
    df = df[df["n"] >= n_min]

    df["moyenne"] = df["diffs"].map(statistics.mean)
    df["ecart_type"] = df["diffs"].map(statistics.stdev)
    df["cv"] = df["ecart_type"] / df["moyenne"]
    return df.reset_index()[["src_ip", "domain", "n", "cv"]]
    # END

In [ ]:
def check_cv_destinations():
    # données à la main : écarts réguliers de 60 s -> CV = 0 ;
    # écarts (10, 30, 20) -> moyenne 20, écart-type 10, CV = 0.5 ;
    # un couple avec un seul écart est écarté (n_min = 3).
    idx = pd.MultiIndex.from_tuples(
        [("a", "x"), ("a", "y"), ("b", "z")], names=["src_ip", "domain"]
    )
    t0 = datetime(2026, 1, 1, 8, 0, 0)
    sec = lambda *s: [t0 + timedelta(seconds=v) for v in s]
    petit = pd.DataFrame(
        {"timestamps": [sec(0, 60, 120, 180, 240), sec(0, 10, 40, 60), sec(0, 5)]},
        index=idx,
    )
    res = cv_destinations(petit, n_min=3)
    assert list(res.columns) == ["src_ip", "domain", "n", "cv"], "colonnes incorrectes"
    res = res.set_index(["src_ip", "domain"])
    assert len(res) == 2, "le couple trop rare doit être écarté"
    assert res.loc[("a", "x"), "n"] == 4
    assert res.loc[("a", "x"), "cv"] == 0.0
    assert res.loc[("a", "y"), "n"] == 3
    assert res.loc[("a", "y"), "cv"] == 0.5
    print("ex 3 OK")

check_cv_destinations()

On affiche ici les couples `(src_ip, domain)` triés par CV croissant.

In [ ]:
cv_destinations(timestamps_par_src_dest(df_logs_reseau)).sort_values("cv")

### 1.4 Liste blanche : les faux positifs périodiques

Le tri par CV ci-dessus montre que les couples les plus réguliers ne sont pas
forcément malveillants : les mises à jour (Windows, antivirus, NTP…) contactent
elles aussi leur serveur à intervalle quasi constant — avec un petit jitter réseau
du même ordre que celui du beacon. Ce sont des **faux positifs**.

L'équipe SOC fournit la liste des domaines légitimes connus (liste blanche) :
on les écarte avant de chercher le beacon. Cette liste est **réutilisée dans la
suite du TD** chaque fois qu'il faut écarter le trafic légitime.

Domaines whitelistés : `windowsupdate.com`, `ntp.org`, `av-updates.example.net`,
`sync-backup.example.net`, `metrics-updates.example.net`.

In [ ]:
# Domaines légitimes périodiques (liste blanche fournie par le SOC).
# À réutiliser chaque fois qu'il faut écarter le trafic légitime.
DOMAINES_WHITELIST = {
    "windowsupdate.com",
    "ntp.org",
    "av-updates.example.net",
    "sync-backup.example.net",
    "metrics-updates.example.net",
}

### 1.5 Détection du beacon

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4</legend>

Complétez `detecter_beacon_timing` : elle renvoie le couple `(src_ip, domain)` de CV minimal, en écartant les domaines de `DOMAINES_WHITELIST` (utiliser `timestamps_par_src_dest` et `cv_destinations`).

</fieldset>

In [ ]:
def detecter_beacon_timing(df: pd.DataFrame) -> tuple[str, str]:
    """Renvoie (src_ip, domain) du beacon le plus régulier, hors liste blanche."""
    # BEGIN
    # raise NotImplementedError
    stats = cv_destinations(timestamps_par_src_dest(df))
    # Certains domaines légitimes (mises à jour, NTP…) sont *aussi* périodiques :
    # on les écarte avant de chercher le CV minimal.
    hors_whitelist = stats[~stats["domain"].isin(DOMAINES_WHITELIST)]
    suspect = hors_whitelist.loc[hors_whitelist["cv"].idxmin()]
    return suspect["src_ip"], suspect["domain"]
    # END

In [ ]:
src, domaine = detecter_beacon_timing(df_logs_reseau)
assert (src, domaine) == ("10.0.0.66", "sync-updates.example.net"), f"trouvé : {src} -> {domaine}"
print("ex 4 OK :", src, "->", domaine)

### 1.6 Extraction des IOC

Le couple (hôte, domaine) suspect étant identifié, on extrait ses **IOC**
(indicateurs de compromission) : les IP de destination distinctes
contactées par cet hôte pour ce domaine.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 5</legend>

Complétez `extraire_ioc_beacon` : elle renvoie un dataframe des destinations distinctes (une ligne par IP de destination, colonnes `dst_ip` et `domain`) contactées par `(src_ip, domain)`, trié par `dst_ip`.

</fieldset>

In [ ]:
def extraire_ioc_beacon(df: pd.DataFrame, src_ip: str, domain: str) -> pd.DataFrame:
    """IOC du beacon : destinations distinctes contactées par `(src_ip, domain)`.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par `dst_ip`.
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df[(df["src_ip"] == src_ip) & (df["domain"] == domain)][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
src, domaine = detecter_beacon_timing(df_logs_reseau)
ioc_beacon = extraire_ioc_beacon(df_logs_reseau, src, domaine)
assert len(ioc_beacon) == 1
assert ioc_beacon.iloc[0]["dst_ip"] == "185.165.29.10"
assert ioc_beacon.iloc[0]["domain"] == "sync-updates.example.net"
print("ex 5 OK :", len(ioc_beacon), "IOC")

## 2. Timing aléatoire : il ne reste que la plage d'IP

L'implant le plus sophistiqué **randomise totalement** son timing : plus aucun
rythme à détecter. Mais il doit toujours joindre son serveur C2, hébergé dans un
`/24` que l'entreprise ne contacte jamais.

**Méthode :**

1. Écarter le domaine du beacon déjà identifié au cas 1
   (`sync-updates.example.net`).
2. Réduire chaque IP restante à son préfixe `/24`.
3. Pour chaque préfixe, calculer **deux chiffres séparés** :
   - `nb_sources` : combien d'hôtes distincts le touchent ;
   - `volume` : combien de connexions il reçoit.
4. Un préfixe **rare** (1 seule source) **et massivement contacté** = suspect.

**Note :** le timing aléatoire ne laisse aucun rythme exploitable :
ni la régularité (cas 1) ni le comptage de destinations (cas 3) ne le détectent.
Seule la plage d'IP le trahit.

### 2.1 Préfixe /24 d'une IP

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 6</legend>

Complétez `prefixe` : elle renvoie le préfixe `/24` d'une IPv4, c'est-à-dire ses trois premiers octets (par exemple `"185.165.29"` pour `"185.165.29.10"`).

</fieldset>

In [ ]:
def prefixe(ip: str) -> str:
    """Préfixe /24 d'une IPv4."""
    # BEGIN
    # raise NotImplementedError
    return ".".join(ip.split(".")[:3])
    # END

In [ ]:
assert prefixe("185.165.29.10") == "185.165.29"
assert prefixe("10.0.0.1") == "10.0.0"
print("ex 6 OK")

In [ ]:
# DELETE
df_logs_reseau.assign(prefix=df_logs_reseau["dst_ip"].map(prefixe))

### 2.2 Hôtes sur une plage rare et massivement contactée

Pour chaque préfixe `/24` des destinations, on compte le nombre de sources
distinctes qui le contactent, ainsi que le nombre total de requêtes qu'il reçoit.
On ne garde ensuite que les préfixes à une seule source et au volume élevé,
puis on remonte aux hôtes sources correspondants.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 7</legend>

Complétez `hotes_plage_rare` : elle renvoie la liste triée des hôtes source qui touchent un `/24` ayant une seule source et au moins `vol_min` connexions, après avoir écarté le domaine `sync-updates.example.net`.

</fieldset>

In [ ]:
def hotes_plage_rare(df: pd.DataFrame, vol_min: int = 50) -> list[str]:
    """Hôtes source touchant un `/24` rare (1 source) ET massivement contacté."""
    # BEGIN
    # raise NotImplementedError
    ## Le beacon du cas 1 est lui aussi sur un /24 rare : on exclut
    ## son domaine, déjà identifié, pour ne garder que le nouvel implant.
    df = df[~df["domain"].isin({"sync-updates.example.net"})].copy()
    df["prefix"] = df["dst_ip"].map(prefixe)

    # deux signaux, calculés séparément :
    nb_sources = df.groupby("prefix")["src_ip"].nunique()  # combien d'hôtes distincts ?
    volume = df.groupby("prefix")["timestamp"].count()      # combien de connexions ?

    rares = nb_sources[(nb_sources == 1) & (volume >= vol_min)].index
    return sorted(df[df["prefix"].isin(rares)]["src_ip"].unique())
    # END

In [ ]:
suspects = hotes_plage_rare(df_logs_reseau)
assert suspects == ["10.0.0.68"], f"trouvé : {suspects}"
print("ex 7 OK :", suspects)

### 2.3 Extraction des IOC

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 8</legend>

Complétez `extraire_ioc_plage` : elle renvoie les destinations distinctes (une ligne par IP de destination, colonnes `dst_ip` et `domain`) contactées par l'hôte suspect, triées par `dst_ip`.

</fieldset>

In [ ]:
def extraire_ioc_plage(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par `dst_ip`.
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
suspect = hotes_plage_rare(df_logs_reseau)[0]
ioc_plage = extraire_ioc_plage(df_logs_reseau, suspect)
assert len(ioc_plage) == 5
assert ioc_plage["dst_ip"].is_unique
assert (ioc_plage["dst_ip"].str.startswith("91.240.118.")).all()
print("ex 8 OK :", len(ioc_plage), "IOC")

## 3. Fast-flux : la destination qui tourne

Certains C2 changent d'IP **et** de domaine à chaque connexion pour échapper aux
filtres : c'est le **fast-flux** (ou DGA). Chaque domaine n'apparaissant qu'une
fois, grouper par domaine est inutile.

**Méthode :** regarder l'hôte *source*. Un hôte légitime contacte un petit nombre
de destinations. Un hôte en fast-flux contacte un grand nombre d'IP **distinctes**.

### 3.1 Hôte aux destinations multiples

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 9</legend>

Complétez `detecter_fastflux` : elle renvoie l'hôte source qui contacte le plus d'IP de destination distinctes.

</fieldset>

In [ ]:
def detecter_fastflux(df: pd.DataFrame) -> str:
    """Renvoie l'hôte source qui contacte le plus d'IP de destination distinctes."""
    # BEGIN
    # raise NotImplementedError
    return df.groupby("src_ip")["dst_ip"].nunique().idxmax()
    # END

In [ ]:
hote = detecter_fastflux(df_logs_reseau)
assert hote == "10.0.0.67", f"trouvé : {hote}"
print("ex 9 OK :", hote)

### 3.2 Extraction des IOC

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 10</legend>

Complétez `extraire_ioc` : elle renvoie les destinations distinctes (une ligne par IP de destination, colonnes `dst_ip` et `domain`) contactées par l'hôte `src_ip`, triées par `dst_ip`.

</fieldset>

In [ ]:
def extraire_ioc(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par `dst_ip`.
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
hote = detecter_fastflux(df_logs_reseau)
ioc = extraire_ioc(df_logs_reseau, hote)
assert len(ioc) == 40
assert ioc["dst_ip"].is_unique
print("ex 10 OK :", len(ioc), "IOC (IP de destination distinctes)")

## 4. Quand le comptage ne suffit plus

Le réseau grossit : on observe désormais une centaine d'hôtes, qui contactent
chacun des centaines d'IP de destination distinctes (CDN, edge, caches…).

Un hôte compromis renouvelle ses destinations à chaque connexion.

### 4.1 Chargement du grand réseau

In [ ]:
df_logs_reseau_grand = pd.read_csv("logs_reseau_grand.csv", parse_dates=["timestamp"])
print(df_logs_reseau_grand.shape)
df_logs_reseau_grand.head()

### 4.2 Détection de l'hôte compromis

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 11</legend>

Complétez `detecter_hote_compromis` : elle renvoie l'hôte compromis du grand réseau.

</fieldset>

In [ ]:
def detecter_hote_compromis(df: pd.DataFrame) -> str:
    """Renvoie l'hôte compromis du grand réseau."""
    # BEGIN
    # raise NotImplementedError
    nb_sources = df.groupby("dst_ip")["src_ip"].nunique()
    ip_rares = nb_sources[nb_sources == 1].index
    rare_par_hote = df[df["dst_ip"].isin(ip_rares)].groupby("src_ip")["dst_ip"].nunique()
    return rare_par_hote.idxmax()
    # END

In [ ]:
victime = detecter_hote_compromis(df_logs_reseau_grand)
assert victime == "10.0.0.167", f"trouvé : {victime}"
print("ex 11 OK :", victime)

### 4.3 Extraction des IOC

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 12</legend>

Complétez `extraire_ioc_grand` : elle renvoie les destinations distinctes (une ligne par IP de destination, colonnes `dst_ip` et `domain`) contactées par l'hôte compromis, triées par `dst_ip`.

</fieldset>

In [ ]:
def extraire_ioc_grand(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par `dst_ip`.
    """
    # BEGIN
    # raise NotImplementedError
    return (
        df[df["src_ip"] == src_ip][["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
victime = detecter_hote_compromis(df_logs_reseau_grand)
ioc_grand = extraire_ioc_grand(df_logs_reseau_grand, victime)
assert len(ioc_grand) == 400
assert ioc_grand["dst_ip"].is_unique
print("ex 12 OK :", len(ioc_grand), "IOC")

## Bilan

| cas d'étude | signal | notion pandas | hôte trouvé |
| --- | --- | --- | --- |
| 1 | régularité temporelle | `groupby` + `agg(list)` + CV | `10.0.0.66` |
| 2 | plage d'IP rare et concentrée | `map` + deux `groupby` | `10.0.0.68` |
| 3 | multiplicité des destinations | `groupby().nunique()` | `10.0.0.67` |
| 4 | destinations rares (une seule source) | `nunique` + `isin` | `10.0.0.167` |

**À retenir :** la détection de C2 est **heuristique** — on croise plusieurs
signaux (rythme, destinations, plage d'IP, rareté des destinations). Aucun
signal seul ne suffit, et chaque contre-mesure de l'attaquant (jitter,
fast-flux, timing aléatoire, destinations partagées) casse un signal… mais en
laisse un autre.